# Notebook 10 — Final Submission

**Goal:** Train LGBM + XGBoost with GroupKFold, apply Platt calibration + Stacking V2 metalearner, generate test predictions, and save the submission CSV.

**Strategy (from Notebook 09):**
- Primary: Stacking V2 (LGBM Platt + XGBoost Platt + raw features: Stint, TyreLife×laps_remaining, RaceProgress)
- Fallback: LGBM solo if stacking V2 CV AUC < 0.8558

**Self-contained:** All functions defined inline. All saved calibrator/stacker coefficients hardcoded. Runs on Kaggle without local file structure.

In [ ]:
# Uncomment when running on Kaggle
# !pip install lightgbm==4.5.0 xgboost==2.1.3 scikit-learn==1.5.2 pandas==2.2.3 numpy==1.26.4 pyarrow==17.0.0 -q

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import pickle
from pathlib import Path
from scipy.special import logit as scipy_logit
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

np.random.seed(42)
print('Imports OK')
print(f'  LightGBM: {lgb.__version__}')
print(f'  XGBoost:  {xgb.__version__}')

Imports OK
  LightGBM: 4.6.0
  XGBoost:  3.2.0


In [2]:
# ── Path detection (works in VSCode, JupyterLab, and Kaggle) ─────────────────
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root  = cwd
raw_dir       = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
models_dir    = project_root / 'models'
sub_dir       = project_root / 'submissions'
sub_dir.mkdir(exist_ok=True)
print(f'Project root: {project_root}')

train_raw = pd.read_csv(raw_dir / 'train.csv')
test_raw  = pd.read_csv(raw_dir / 'test.csv')
sample_sub = pd.read_csv(raw_dir / 'sample_submission.csv')
print(f'Train: {train_raw.shape}  |  Test: {test_raw.shape}  |  Sample sub: {sample_sub.shape}')

Project root: c:\Repos\predict-f1-pit-stops
Train: (439140, 16)  |  Test: (188165, 15)  |  Sample sub: (188165, 2)


In [3]:
# ── Feature engineering (exact replica of Notebook 02) ───────────────────────

def build_base_features(df: pd.DataFrame) -> pd.DataFrame:
    CLIFF   = {'SOFT': 13, 'MEDIUM': 49, 'HARD': 61, 'INTERMEDIATE': 1, 'WET': 1}
    ORDINAL = {'SOFT':  1, 'MEDIUM':  2, 'HARD':  3, 'INTERMEDIATE': 0, 'WET': 0}
    W_LO, W_HI = -205, 122
    STINT_KEYS = ['Race', 'Year', 'Driver', 'Stint']

    df = df.copy().sort_values(['Race', 'Year', 'LapNumber']).reset_index(drop=True)

    # Tyre age
    df['TyreLife_normalized_by_compound'] = df['TyreLife'] / df['Compound'].map(CLIFF)
    df['TyreLife_sq']                     = df['TyreLife'] ** 2

    # Compound encoding
    df['is_wet_tyre']      = df['Compound'].isin({'INTERMEDIATE', 'WET'}).astype(np.int8)
    df['compound_ordinal'] = df['Compound'].map(ORDINAL).astype(np.int8)

    # Race context
    df['laps_remaining']     = 1.0 - df['RaceProgress']
    df['is_testing_session'] = (df['Race'] == 'Pre-Season Testing').astype(np.int8)

    # Degradation
    df['Cumulative_Degradation_winsorized'] = df['Cumulative_Degradation'].clip(W_LO, W_HI)
    df['Degradation_rate']        = df['Cumulative_Degradation_winsorized'] / df['TyreLife'].clip(lower=1)
    df['Degradation_acceleration'] = (
        df.groupby(STINT_KEYS)['Cumulative_Degradation']
          .diff(1)
          .fillna(0.0)
    )

    # Lag features
    for raw_col, feat_base in [
        ('LapTime (s)',   'LapTime'),
        ('LapTime_Delta', 'LapTime_Delta'),
    ]:
        grp = df.groupby(STINT_KEYS)[raw_col]
        for lag in [1, 2, 3]:
            df[f'{feat_base}_lag{lag}'] = grp.shift(lag).fillna(0.0)

    # Rolling stats
    for w in [3, 5]:
        df[f'LapTime_rolling_mean_{w}'] = (
            df.groupby(STINT_KEYS)['LapTime (s)']
              .rolling(w, min_periods=1)
              .mean()
              .droplevel(list(range(len(STINT_KEYS))))
              .sort_index()
        )
        df[f'LapTime_rolling_std_{w}'] = (
            df.groupby(STINT_KEYS)['LapTime (s)']
              .rolling(w, min_periods=2)
              .std()
              .droplevel(list(range(len(STINT_KEYS))))
              .sort_index()
              .fillna(0.0)
        )

    deg_grp = df.groupby(STINT_KEYS)['Cumulative_Degradation']
    for w in [3, 5]:
        slope = (df['Cumulative_Degradation'] - deg_grp.shift(w - 1)) / (w - 1)
        df[f'Degradation_rolling_slope_{w}'] = slope.fillna(0.0)

    # Interactions
    df['TyreLife_x_laps_remaining']   = df['TyreLife'] * df['laps_remaining']
    df['Degradation_x_RaceProgress']  = df['Cumulative_Degradation_winsorized'] * df['RaceProgress']
    df['Position_x_RaceProgress']     = df['Position'] * df['RaceProgress']

    return df


def apply_target_encodings(train_df: pd.DataFrame, encode_df: pd.DataFrame) -> pd.DataFrame:
    global_mean = train_df['PitNextLap'].mean()
    result = encode_df.copy()

    race_enc = train_df.groupby('Race')['PitNextLap'].mean()
    result['Race_target_encoded'] = result['Race'].map(race_enc).fillna(global_mean)

    driver_enc = train_df.groupby('Driver')['PitNextLap'].mean()
    result['Driver_target_encoded'] = result['Driver'].map(driver_enc).fillna(global_mean)

    driver_stints = (
        train_df
        .groupby(['Driver', 'Race', 'Year', 'Stint'])['TyreLife']
        .max()
        .reset_index()
        .groupby('Driver')['TyreLife']
        .mean()
    )
    global_avg_stint = driver_stints.mean()
    result['Driver_avg_stint_length'] = result['Driver'].map(driver_stints).fillna(global_avg_stint)

    return result


print('Feature functions defined.')

Feature functions defined.


In [4]:
# ── Apply feature engineering ─────────────────────────────────────────────────
print('Building train features...')
train = build_base_features(train_raw)
print(f'  Train: {train.shape}')

print('Building test features...')
test = build_base_features(test_raw)
print(f'  Test:  {test.shape}')

# GroupKFold groups: entire race-year goes to one fold
groups = train['Race'].astype(str) + '_' + train['Year'].astype(str)
print(f'Unique race-year groups: {groups.nunique()}')

Building train features...
  Train: (439140, 40)
Building test features...
  Test:  (188165, 39)
Unique race-year groups: 104


In [8]:
# ── Hardcoded params and calibrator coefficients ──────────────────────────────

FULL_FEATURES = [
    'TyreLife_normalized_by_compound', 'TyreLife_sq',
    'is_wet_tyre', 'compound_ordinal',
    'laps_remaining', 'is_testing_session',
    'Stint', 'Position',
    'Cumulative_Degradation_winsorized', 'Degradation_rate', 'Degradation_acceleration',
    'LapTime_lag1', 'LapTime_lag2', 'LapTime_lag3',
    'LapTime_Delta_lag1', 'LapTime_Delta_lag2', 'LapTime_Delta_lag3',
    'LapTime_rolling_mean_3', 'LapTime_rolling_mean_5',
    'LapTime_rolling_std_3',  'LapTime_rolling_std_5',
    'Degradation_rolling_slope_3', 'Degradation_rolling_slope_5',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
    'Race_target_encoded', 'Driver_target_encoded', 'Driver_avg_stint_length',
]

# LGBM Optuna-tuned params (from Notebook 05)
LGBM_PARAMS = dict(
    n_estimators      = 2000,
    learning_rate     = 0.049227921561742695,
    num_leaves        = 62,
    min_child_samples = 91,
    subsample         = 0.7528706019183118,
    colsample_bytree  = 0.7463883150642037,
    reg_alpha         = 9.791677886233918,
    reg_lambda        = 1.8738414904490494e-07,
    random_state      = 42,
    verbose           = -1,
)

# XGBoost params (from Notebook 05)
XGB_PARAMS = dict(
    objective             = 'binary:logistic',
    tree_method           = 'hist',
    n_estimators          = 2000,
    early_stopping_rounds = 50,
    learning_rate         = 0.05,
    max_depth             = 6,
    min_child_weight      = 20,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    eval_metric           = 'auc',
    random_state          = 42,
    verbosity             = 0,
)

# Platt calibrator coefficients (from Notebook 08)
# Input: scipy_logit(clip(raw_pred))  →  output: sigmoid(coef * logit + intercept)
PLATT_LGBM = dict(coef=0.9685489,  intercept=0.00375338)
PLATT_XGB  = dict(coef=1.05463229, intercept=0.13574348)

# Stacking V2 metalearner coefficients (from Notebook 09)
# Features: [lgbm_platt, xgb_platt, Stint_scaled, TyreLife_x_laps_rem_scaled, RaceProgress_scaled]
SCALER_V2_MEAN  = np.array([1.78911281, 7.8318298,  0.33766054])   # [Stint, TyreLife_x_laps_rem, RaceProgress]
SCALER_V2_SCALE = np.array([0.95019306, 4.70797922, 0.25327721])
STACK_V2_COEF   = np.array([4.21093403, 0.97707202, 0.39730814, 0.38486505, -0.09216163])
STACK_V2_INTER  = -2.87682601

LGBM_SOLO_AUC_THRESHOLD = 0.8558   # fall back to LGBM solo if stacking V2 CV AUC < this

print('Params and calibrator coefficients loaded.')
print(f'  FULL_FEATURES: {len(FULL_FEATURES)} features')

Params and calibrator coefficients loaded.
  FULL_FEATURES: 29 features


In [9]:
# ── Helper functions ──────────────────────────────────────────────────────────

def platt_scale(raw_pred: np.ndarray, coef: float, intercept: float) -> np.ndarray:
    logit_p = scipy_logit(np.clip(raw_pred, 1e-7, 1 - 1e-7))
    return 1.0 / (1.0 + np.exp(-(coef * logit_p + intercept)))


def stack_v2_predict(lgbm_platt: np.ndarray, xgb_platt: np.ndarray,
                     stint: np.ndarray, tyre_x_laps_rem: np.ndarray,
                     race_progress: np.ndarray) -> np.ndarray:
    raw = np.column_stack([stint, tyre_x_laps_rem, race_progress])
    raw_scaled = (raw - SCALER_V2_MEAN) / SCALER_V2_SCALE
    X = np.column_stack([lgbm_platt, xgb_platt, raw_scaled])
    return 1.0 / (1.0 + np.exp(-(X @ STACK_V2_COEF + STACK_V2_INTER)))


print('Helper functions defined.')

Helper functions defined.


In [10]:
# ── GroupKFold training loop ──────────────────────────────────────────────────
import time

N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)

n_train = len(train)
n_test  = len(test)

oof_lgbm = np.zeros(n_train)
oof_xgb  = np.zeros(n_train)
test_lgbm_folds = np.zeros((n_test, N_FOLDS))
test_xgb_folds  = np.zeros((n_test, N_FOLDS))

fold_aucs_lgbm = []
fold_aucs_xgb  = []

t0 = time.time()

for fold_idx, (tr_idx, val_idx) in enumerate(gkf.split(train, train['PitNextLap'], groups=groups)):
    print(f'\n── Fold {fold_idx} ({len(tr_idx):,} train / {len(val_idx):,} val) ──')

    # Apply target encodings inside fold (no leakage)
    tr_df       = apply_target_encodings(train.iloc[tr_idx], train.iloc[tr_idx])
    val_df      = apply_target_encodings(train.iloc[tr_idx], train.iloc[val_idx])
    test_df_enc = apply_target_encodings(train.iloc[tr_idx], test)

    X_tr,  y_tr  = tr_df[FULL_FEATURES].to_numpy(),  tr_df['PitNextLap'].to_numpy()
    X_val, y_val = val_df[FULL_FEATURES].to_numpy(), val_df['PitNextLap'].to_numpy()
    X_test       = test_df_enc[FULL_FEATURES].to_numpy()

    # ── LightGBM ──
    lgbm_model = lgb.LGBMClassifier(**LGBM_PARAMS)
    lgbm_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    )
    oof_lgbm[val_idx]            = lgbm_model.predict_proba(X_val)[:, 1]
    test_lgbm_folds[:, fold_idx] = lgbm_model.predict_proba(X_test)[:, 1]
    auc_lgbm = roc_auc_score(y_val, oof_lgbm[val_idx])
    fold_aucs_lgbm.append(auc_lgbm)
    print(f'  LGBM : AUC={auc_lgbm:.4f}  trees={lgbm_model.best_iteration_}')

    # ── XGBoost ──
    xgb_model = xgb.XGBClassifier(**XGB_PARAMS)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx]            = xgb_model.predict_proba(X_val)[:, 1]
    test_xgb_folds[:, fold_idx] = xgb_model.predict_proba(X_test)[:, 1]
    auc_xgb = roc_auc_score(y_val, oof_xgb[val_idx])
    fold_aucs_xgb.append(auc_xgb)
    best_xgb_trees = xgb_model.best_iteration if hasattr(xgb_model, 'best_iteration') else xgb_model.n_estimators
    print(f'  XGB  : AUC={auc_xgb:.4f}  trees={best_xgb_trees}')

print(f'\n── Training complete ({time.time() - t0:.0f}s) ──')
print(f'LGBM per-fold AUCs: {[f"{a:.4f}" for a in fold_aucs_lgbm]}')
print(f'XGB  per-fold AUCs: {[f"{a:.4f}" for a in fold_aucs_xgb]}')

cv_auc_lgbm = roc_auc_score(train['PitNextLap'], oof_lgbm)
cv_auc_xgb  = roc_auc_score(train['PitNextLap'], oof_xgb)
print(f'\nOOF CV AUC — LGBM: {cv_auc_lgbm:.4f}  |  XGB: {cv_auc_xgb:.4f}')

# Average test predictions across folds
test_lgbm = test_lgbm_folds.mean(axis=1)
test_xgb  = test_xgb_folds.mean(axis=1)


── Fold 0 (351,122 train / 88,018 val) ──


c:\Repos\predict-f1-pit-stops\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Repos\predict-f1-pit-stops\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  LGBM : AUC=0.8686  trees=284
  XGB  : AUC=0.8653  trees=632

── Fold 1 (351,696 train / 87,444 val) ──


c:\Repos\predict-f1-pit-stops\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Repos\predict-f1-pit-stops\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  LGBM : AUC=0.8246  trees=35
  XGB  : AUC=0.8503  trees=11

── Fold 2 (351,113 train / 88,027 val) ──


...

  LGBM : AUC=0.8404  trees=41
  XGB  : AUC=0.8394  trees=16

── Fold 3 (351,286 train / 87,854 val) ──


c:\Repos\predict-f1-pit-stops\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Repos\predict-f1-pit-stops\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  LGBM : AUC=0.8893  trees=462
  XGB  : AUC=0.8877  trees=413

── Fold 4 (351,343 train / 87,797 val) ──


c:\Repos\predict-f1-pit-stops\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Repos\predict-f1-pit-stops\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  LGBM : AUC=0.8670  trees=294
  XGB  : AUC=0.8686  trees=448

── Training complete (92s) ──
LGBM per-fold AUCs: ['0.8686', '0.8246', '0.8404', '0.8893', '0.8670']
XGB  per-fold AUCs: ['0.8653', '0.8503', '0.8394', '0.8877', '0.8686']

OOF CV AUC — LGBM: 0.8558  |  XGB: 0.8492


In [11]:
# ── Platt calibration ─────────────────────────────────────────────────────────
# OOF
oof_lgbm_platt = platt_scale(oof_lgbm, **PLATT_LGBM)
oof_xgb_platt  = platt_scale(oof_xgb,  **PLATT_XGB)

# Test
test_lgbm_platt = platt_scale(test_lgbm, **PLATT_LGBM)
test_xgb_platt  = platt_scale(test_xgb,  **PLATT_XGB)

# AUC is rank-invariant — calibration doesn't change it, but verify
assert abs(roc_auc_score(train['PitNextLap'], oof_lgbm_platt) - cv_auc_lgbm) < 1e-8
print('Platt calibration applied. AUC invariance confirmed.')

Platt calibration applied. AUC invariance confirmed.


In [12]:
# ── Stacking V2 ───────────────────────────────────────────────────────────────
# Stacker V2 features: [lgbm_platt, xgb_platt, Stint_scaled, TyreLife_x_laps_rem_scaled, RaceProgress_scaled]
# Note: the 3rd raw feature is RaceProgress (not laps_remaining) — matches how scaler was fit in NB09.

# OOF raw features (train rows, original order preserved by iloc)
oof_stint        = train['Stint'].to_numpy()
oof_tyre_x_laps  = train['TyreLife_x_laps_remaining'].to_numpy()
oof_race_prog    = train['RaceProgress'].to_numpy()

# Test raw features
test_stint       = test['Stint'].to_numpy()
test_tyre_x_laps = test['TyreLife_x_laps_remaining'].to_numpy()
test_race_prog   = test['RaceProgress'].to_numpy()

# Generate stacking V2 predictions
oof_stack_v2  = stack_v2_predict(oof_lgbm_platt,  oof_xgb_platt,
                                  oof_stint,        oof_tyre_x_laps,  oof_race_prog)
test_stack_v2 = stack_v2_predict(test_lgbm_platt, test_xgb_platt,
                                  test_stint,       test_tyre_x_laps, test_race_prog)

cv_auc_stack_v2 = roc_auc_score(train['PitNextLap'], oof_stack_v2)
print(f'CV AUC — LGBM solo:   {cv_auc_lgbm:.4f}')
print(f'CV AUC — Stacking V2: {cv_auc_stack_v2:.4f}  (delta={cv_auc_stack_v2 - cv_auc_lgbm:+.4f})')

CV AUC — LGBM solo:   0.8558
CV AUC — Stacking V2: 0.8569  (delta=+0.0011)


In [13]:
# ── Strategy selection ────────────────────────────────────────────────────────
if cv_auc_stack_v2 > LGBM_SOLO_AUC_THRESHOLD:
    strategy      = 'stacking_v2'
    test_final    = test_stack_v2
    final_cv_auc  = cv_auc_stack_v2
    print(f'Strategy: STACKING V2  (CV AUC={final_cv_auc:.4f} > threshold {LGBM_SOLO_AUC_THRESHOLD})')
else:
    strategy      = 'lgbm_solo'
    test_final    = test_lgbm_platt
    final_cv_auc  = cv_auc_lgbm
    print(f'Strategy: LGBM SOLO  (stacking V2={cv_auc_stack_v2:.4f} <= threshold — fell back)')

print(f'Final CV AUC: {final_cv_auc:.4f}')

Strategy: STACKING V2  (CV AUC=0.8569 > threshold 0.8558)
Final CV AUC: 0.8569


In [14]:
# ── Submission validation ─────────────────────────────────────────────────────
assert len(test_final) == 188165, f'Row count mismatch: {len(test_final)}'
assert set(test['id']) == set(sample_sub['id']), 'id mismatch vs sample_submission'
assert np.all((test_final >= 0) & (test_final <= 1)), 'Predictions out of [0,1]'
assert not np.any(np.isnan(test_final)), 'NaN predictions found'

print('All validation checks passed.')
print(f'  Rows:        {len(test_final):,}')
print(f'  Pred range:  [{test_final.min():.4f}, {test_final.max():.4f}]')
print(f'  Pred mean:   {test_final.mean():.4f}')

All validation checks passed.
  Rows:        188,165
  Pred range:  [0.0188, 0.9723]
  Pred mean:   0.2032


In [15]:
# ── Save submission ───────────────────────────────────────────────────────────
submission = test[['id']].copy()
submission['PitNextLap'] = test_final

# Align to sample_submission row order
submission = sample_sub[['id']].merge(submission, on='id', how='left')

sub_filename = f'submission_v001_{strategy}.csv'
sub_path = sub_dir / sub_filename
submission.to_csv(sub_path, index=False)
print(f'Saved: {sub_path}')
print(submission.head(3))

Saved: c:\Repos\predict-f1-pit-stops\submissions\submission_v001_stacking_v2.csv
       id  PitNextLap
0  439140    0.065186
1  439141    0.089011
2  439142    0.064305


In [16]:
# ── Log to leaderboard_log.md ─────────────────────────────────────────────────
import datetime
log_path = sub_dir / 'leaderboard_log.md'

today = datetime.date.today().isoformat()
log_entry = (
    f'| {sub_filename} | {strategy} | {final_cv_auc:.4f} | — | '
    f'Submitted {today}; LB AUC TBD after Kaggle upload |\n'
)

if not log_path.exists():
    header = (
        '| File | Strategy | CV AUC | LB AUC | Notes |\n'
        '|------|----------|--------|--------|-------|\n'
    )
    log_path.write_text(header + log_entry)
else:
    with open(log_path, 'a') as f:
        f.write(log_entry)

print(f'Logged to {log_path}')
print(log_entry.strip())

Logged to c:\Repos\predict-f1-pit-stops\submissions\leaderboard_log.md
| submission_v001_stacking_v2.csv | stacking_v2 | 0.8569 | — | Submitted 2026-05-20; LB AUC TBD after Kaggle upload |
